In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.calibration import CalibratedClassifierCV
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.metrics import log_loss, roc_auc_score, brier_score_loss, classification_report
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

In [ ]:
path = r"C:\Users\bhavy\Massachusetts Institute of Technology\Truck Parking Capstone - General\Truck Stop Finder 🚚⛽\\"
# path = r"C:\Users\samcl\Massachusetts Institute of Technology\Truck Parking Capstone - Truck Stop Finder 🚚⛽\\"

In [ ]:
# Sourced directly from TruckerPath
park_data_1 = pd.read_csv(
    path + r"5. Source & Refrence Files\0. TruckerPath Data\MIT_2025_High_Volume_Routes_Parking_Data_1 - Copy.csv")
park_data_2 = pd.read_csv(
    path + r"5. Source & Refrence Files\0. TruckerPath Data\MIT_2025_High_Volume_Routes_Parking_Data_2 - Copy.csv")
park_data_3 = pd.read_csv(
    path + r"5. Source & Refrence Files\0. TruckerPath Data\MIT_2025_High_Volume_Routes_Parking_Data_3 - Copy.csv")
park_data = pd.concat([park_data_1, park_data_2, park_data_3], ignore_index=True)

In [ ]:
truck_stop = pd.read_excel("output_excel\Model_Stops_V3.xlsx")

In [ ]:
avail_park = park_data[park_data["pin id"].isin(truck_stop["pin id"].unique())].copy()

In [ ]:
avail_park["ts_utc"] = pd.to_datetime(avail_park["time(utc)"], utc=True)

In [ ]:
print(avail_park["ts_utc"].isna().mean(), "fraction of timestamps failed to parse")
print(avail_park["ts_utc"].min(), "to", avail_park["ts_utc"].max())

In [ ]:
# avail_park["day_of_week"] = avail_park["time(utc)"].dt.dayofweek
# avail_park["day_name"] = avail_park["time(utc)"].dt.day_name()
# avail_park["hour"] = avail_park["time(utc)"].dt.hour
# avail_park["month"] = avail_park["time(utc)"].dt.month
# avail_park["date"] = avail_park["time(utc)"].dt.day

In [ ]:
avail_park = avail_park[avail_park["parking status"] != 'Paid'].copy()
#TODO: Remove Paid filter

In [ ]:
avail_park.head(2)

In [ ]:
avail_park = avail_park.sort_values(["pin id", "ts_utc"])

In [ ]:
avail_park["gap_prev"] = avail_park.groupby("pin id")["ts_utc"].diff()
avail_park["gap_prev_min"] = avail_park["gap_prev"].dt.total_seconds() / 60

In [ ]:
avail_park

In [ ]:
print(avail_park["gap_prev_min"].describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

In [ ]:
gap_by_stop = (
    avail_park.dropna(subset=["gap_prev_min"])
    .groupby("pin id")["gap_prev_min"]
    .median()
    .sort_values(ascending=False)
)

print("\nTop 10 sparsest stops by median gap (minutes):")
print(gap_by_stop.head(10))

In [ ]:
status_map = {
    "Full": 0,
    "Some": 1,
    "Lots": 2
}

avail_park["status_ord"] = avail_park["parking status"].map(status_map)

In [ ]:
LABEL_TOL = pd.Timedelta("60min")  # ±60 min for label lookup
STALE_CUTOFF = pd.Timedelta("6h")  # 6h staleness rule for last_known_status

obs = avail_park[["pin id", "ts_utc", "status_ord", "parking status"]].dropna(subset=["ts_utc", "status_ord"])
# obs = obs.sort_values(["pin id", "ts_utc"])
obs_sorted = obs.dropna(subset=["ts_utc"]).sort_values(["ts_utc", "pin id"]).reset_index(drop=True)

In [ ]:
obs

In [ ]:
# ---- 2) Helper A: last observation at or before query time (decision-time features)
def attach_last_obs_before(query_df, time_col="query_ts"):
    """
    query_df must have columns: ['pin id', time_col]
    Returns query_df + last observed status at/before query_ts and staleness.
    """
    q = query_df.copy()
    q[time_col] = pd.to_datetime(q[time_col], utc=True, errors="coerce")
    # q = q.sort_values(["pin id", time_col], ignore_index=True)
    q = q.dropna(subset=[time_col]).sort_values([time_col, "pin id"]).reset_index(drop=True)

    out = pd.merge_asof(
        q,
        obs_sorted.rename(
            columns={"ts_utc": "last_ts", "status_ord": "last_status_ord", "parking status": "last_status_txt"}),
        left_on=time_col,
        right_on="last_ts",
        by="pin id",
        direction="backward",  # <= query time
        allow_exact_matches=True
    )

    out["time_since_last_obs_min"] = (out[time_col] - out["last_ts"]).dt.total_seconds() / 60

    # Apply 6h staleness rule: if too old, treat last_known_status as unknown (but keep staleness numeric)
    too_stale = (out[time_col] - out["last_ts"]) > STALE_CUTOFF
    out.loc[too_stale, ["last_status_ord", "last_status_txt", "last_ts"]] = pd.NA

    return out


# ---- 3) Helper B: nearest observation within ±60 minutes (labels around ETA)
def attach_label_nearest(query_df, time_col="eta_ts"):
    """
    query_df must have columns: ['pin id', time_col]
    Returns query_df + label status closest to eta_ts within ±60 minutes.
    If no observation within tolerance, label fields stay NA.
    """
    q = query_df.copy()
    q[time_col] = pd.to_datetime(q[time_col], utc=True, errors="coerce")
    # q = q.sort_values(["pin id", time_col])
    q = q.dropna(subset=[time_col]).sort_values([time_col, "pin id"]).reset_index(drop=True)

    out = pd.merge_asof(
        q,
        obs_sorted.rename(
            columns={"ts_utc": "label_ts", "status_ord": "label_status_ord", "parking status": "label_status_txt"}),
        left_on=time_col,
        right_on="label_ts",
        by="pin id",
        direction="nearest",
        tolerance=LABEL_TOL,
        allow_exact_matches=True
    )

    out["label_time_error_min"] = (out["label_ts"] - out[time_col]).abs().dt.total_seconds() / 60
    return out


In [ ]:
test_q = avail_park[["pin id", "ts_utc"]].dropna().sample(5, random_state=0)
test_q = test_q.rename(columns={"ts_utc": "query_ts"})
test_q["query_ts"] = test_q["query_ts"] + pd.Timedelta("17min")

In [ ]:
print("=== Last obs before (features) ===")
attach_last_obs_before(test_q, time_col="query_ts")

In [ ]:
test_eta = test_q.copy()
test_eta["eta_ts"] = test_eta["query_ts"] + pd.Timedelta("60min")
#TODO: Change to realistic

print("\n=== Nearest within ±60min (label) ===")
attach_label_nearest(test_eta[["pin id", "eta_ts"]], time_col="eta_ts")

In [ ]:
# ---- Settings for the first batch
N_DECISIONS = 500_000  # start here; later you can go 500k / 1M
MIN_TAU = 15  # min travel time in minutes
MAX_TAU = 180  # max travel time in minutes

# 1) Build a "decision pool" from real observed timestamps
#    We sample from actual rows to get realistic t0 distribution.
decision_pool = avail_park[[
    "pin id", "ts_utc", "pinlat", "pinlon", "city", "route_num", "object"
]].dropna(subset=["pin id", "ts_utc"]).copy()

decisions = decision_pool.sample(N_DECISIONS, random_state=0).rename(columns={"ts_utc": "t_obs"}).copy()

In [ ]:
# 2) Sample travel time tau (minutes)
rng = np.random.default_rng(42)
decisions["tau_min"] = rng.integers(MIN_TAU, MAX_TAU + 1, size=len(decisions))
decisions["t0"] = decisions["t_obs"] + pd.to_timedelta(
    rng.integers(1, 46, size=len(decisions)), unit="m"
)

# 3) Compute ETA
decisions["eta_ts"] = decisions["t0"] + pd.to_timedelta(decisions["tau_min"], unit="m")

# 4) Add ETA calendar features (these are allowed; you know ETA at decision time)
decisions["eta_hour"] = decisions["eta_ts"].dt.hour
decisions["eta_day_of_week"] = decisions["eta_ts"].dt.dayofweek  # Monday=0
decisions["eta_month"] = decisions["eta_ts"].dt.month

In [ ]:
decisions.head(3)

In [ ]:


# 5) Attach last observation at/before t0 (features)
feat = attach_last_obs_before(
    decisions.rename(columns={"t0": "query_ts"}),
    time_col="query_ts"
).rename(columns={"query_ts": "t0"})

# 6) Attach label near ETA within ±60 minutes
lab = attach_label_nearest(
    feat[["pin id", "eta_ts"]].copy(),
    time_col="eta_ts"
)

# Merge labels back (by row order, since we kept same ordering)
feat["label_ts"] = lab["label_ts"].values
feat["label_status_ord"] = lab["label_status_ord"].values
feat["label_status_txt"] = lab["label_status_txt"].values
feat["label_time_error_min"] = lab["label_time_error_min"].values

# 7) Keep only rows with valid labels
train_batch = feat.dropna(subset=["label_status_ord"]).copy()

print("Decision rows:", len(decisions))
print("Training rows with labels:", len(train_batch))
print("Label coverage:", round(len(train_batch) / len(decisions), 3))

# 8) Keep just the columns we care about for modeling (first version)
train_batch = train_batch[[
    "pin id", "pinlat", "pinlon", "city", "route_num", "object",
    "t0", "tau_min", "eta_ts", "eta_hour", "eta_day_of_week", "eta_month",
    "last_status_ord", "time_since_last_obs_min",
    "label_status_ord", "label_time_error_min"
]].reset_index(drop=True)

train_batch.head()


In [ ]:
train_batch["time_since_last_obs_min"].describe()

In [ ]:
print(train_batch["last_status_ord"].value_counts(dropna=False))

In [ ]:
n_all = avail_park["pin id"].nunique()
n_train = train_batch["pin id"].nunique()
print("Stops in raw data:", n_all)
print("Stops in training batch:", n_train)
print("Coverage:", n_train / n_all)


In [ ]:
df = train_batch.copy()

In [ ]:
df["y_full"] = (df["label_status_ord"] == 0).astype(int)
df = df.sort_values("t0").reset_index(drop=True)
cut = df["t0"].quantile(0.80)  # last 20% time as test (you can change later)
train_df = df[df["t0"] <= cut].copy()
test_df = df[df["t0"] > cut].copy()

In [ ]:
print("Train rows:", len(train_df), "Test rows:", len(test_df))
print("Train Full rate:", train_df["y_full"].mean().round(3), "Test Full rate:", test_df["y_full"].mean().round(3))
print("Time split cutoff:", cut)

In [ ]:
num_features = ["last_status_ord", "time_since_last_obs_min", "eta_hour", "eta_day_of_week", "eta_month"]
cat_features = ["route_num", "object"]  # add "city" later if you want

In [ ]:
X_train = train_df[num_features + cat_features]
y_train = train_df["y_full"]

X_test = test_df[num_features + cat_features]
y_test = test_df["y_full"]

In [ ]:
numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

In [ ]:
preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_features),
        ("cat", categorical_pipe, cat_features)
    ],
    remainder="drop"
)

In [ ]:
clf = LogisticRegression(
    max_iter=2000,
    # class_weight="balanced",   # helps if Full is rarer
    solver="lbfgs"
)

In [ ]:
base_model = Pipeline(steps=[
    ("prep", preprocess),
    ("clf", clf)
])

In [ ]:
# ---- Create a calibration split inside train (time-based)
train_df = train_df.sort_values("t0").reset_index(drop=True)
cal_cut = train_df["t0"].quantile(0.80)  # last 20% of TRAIN becomes calibration set

fit_df = train_df[train_df["t0"] <= cal_cut].copy()
cal_df = train_df[train_df["t0"] > cal_cut].copy()

X_fit, y_fit = fit_df[num_features + cat_features], fit_df["y_full"]
X_cal, y_cal = cal_df[num_features + cat_features], cal_df["y_full"]

print("\nFit rows:", len(fit_df), "Cal rows:", len(cal_df))
print("Fit Full rate:", round(y_fit.mean(), 3), "Cal Full rate:", round(y_cal.mean(), 3))
print("Calibration cutoff:", cal_cut)

In [ ]:
# ---- 5) Train
base_model.fit(X_fit, y_fit)

calibrated_model = CalibratedClassifierCV(base_model, method="sigmoid", cv="prefit")
calibrated_model.fit(X_cal, y_cal)

In [ ]:
# ---- Evaluate on test
p_base = np.full_like(y_test, fill_value=y_train.mean(), dtype=float)

p_uncal = base_model.predict_proba(X_test)[:, 1]
p_cal = calibrated_model.predict_proba(X_test)[:, 1]

print("\nTEST metrics:")
print("Baseline  LogLoss:", round(log_loss(y_test, p_base), 4), "Brier:", round(brier_score_loss(y_test, p_base), 4),
      "AUC:", round(roc_auc_score(y_test, p_base), 4))
print("Uncal LR  LogLoss:", round(log_loss(y_test, p_uncal), 4), "Brier:", round(brier_score_loss(y_test, p_uncal), 4),
      "AUC:", round(roc_auc_score(y_test, p_uncal), 4))
print("Cal LR    LogLoss:", round(log_loss(y_test, p_cal), 4), "Brier:", round(brier_score_loss(y_test, p_cal), 4),
      "AUC:", round(roc_auc_score(y_test, p_cal), 4))

In [ ]:
# Optional: confusion matrix at threshold 0.5 for calibrated probs
pred = (p_cal >= 0.5).astype(int)
print("\nConfusion matrix @0.5 (calibrated):")
print(confusion_matrix(y_test, pred))
print("\nClassification report @0.5 (calibrated):")
print(classification_report(y_test, pred, digits=3))

In [ ]:
# baseline probability = always predict train Full rate
p_base = np.full_like(y_test, fill_value=y_train.mean(), dtype=float)

print("Baseline log loss:", log_loss(y_test, p_base))
print("Baseline Brier   :", brier_score_loss(y_test, p_base))


In [ ]:
xgb = XGBClassifier(
    n_estimators=400,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    min_child_weight=1.0,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

In [ ]:
base_model = Pipeline(steps=[("prep", preprocess), ("clf", xgb)])

In [ ]:
base_model.fit(X_fit, y_fit)

In [ ]:
cal_model = CalibratedClassifierCV(base_model, method="sigmoid", cv="prefit")
cal_model.fit(X_cal, y_cal)

In [ ]:
# ---------- 3) Evaluate ----------
p_uncal = base_model.predict_proba(X_test)[:, 1]
p_cal = cal_model.predict_proba(X_test)[:, 1]

print("Time cutoff (train/test):", cut)
print("Calibration cutoff (fit/cal):", cal_cut)
print("\nTEST metrics:")
print("Baseline  LogLoss:", round(log_loss(y_test, p_base), 4),
      "Brier:", round(brier_score_loss(y_test, p_base), 4),
      "AUC:", round(roc_auc_score(y_test, p_base), 4))
print("XGB uncal LogLoss:", round(log_loss(y_test, p_uncal), 4),
      "Brier:", round(brier_score_loss(y_test, p_uncal), 4),
      "AUC:", round(roc_auc_score(y_test, p_uncal), 4))
print("XGB cal   LogLoss:", round(log_loss(y_test, p_cal), 4),
      "Brier:", round(brier_score_loss(y_test, p_cal), 4),
      "AUC:", round(roc_auc_score(y_test, p_cal), 4))

In [ ]:

# ---------- 4) Calibration curve plot (reliability diagram) ----------
# Using sklearn.calibration.calibration_curve (like the link you shared)
n_bins = 10
frac_pos_uncal, mean_pred_uncal = calibration_curve(y_test, p_uncal, n_bins=n_bins, strategy="uniform")
frac_pos_cal, mean_pred_cal = calibration_curve(y_test, p_cal, n_bins=n_bins, strategy="uniform")

plt.figure(figsize=(7, 6))
# Perfect calibration line
plt.plot([0, 1], [0, 1], linestyle="--")

# Model curves
plt.plot(mean_pred_uncal, frac_pos_uncal, marker="o", label="XGBoost (uncalibrated)")
plt.plot(mean_pred_cal, frac_pos_cal, marker="o", label="XGBoost (calibrated)")

plt.title("Calibration Curve (Reliability Diagram)")
plt.xlabel("Mean Predicted Probability (bin)")
plt.ylabel("Fraction of Positives (bin)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Optional: histogram of predicted probabilities (helps explain threshold behavior)
plt.figure(figsize=(7, 4))
plt.hist(p_uncal, bins=30, alpha=0.6, label="uncalibrated")
plt.hist(p_cal, bins=30, alpha=0.6, label="calibrated")
plt.title("Predicted Probability Distribution")
plt.xlabel("Predicted P(Full at ETA)")
plt.ylabel("Count")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
thresholds = np.arange(0.05, 0.41, 0.05)

rows = []
for t in thresholds:
    pred = (p_uncal >= t).astype(int)
    tp = ((pred == 1) & (y_test == 1)).sum()
    fp = ((pred == 1) & (y_test == 0)).sum()
    fn = ((pred == 0) & (y_test == 1)).sum()

    recall = tp / (tp + fn)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0

    rows.append((t, recall, precision))

pd.DataFrame(rows, columns=["threshold", "recall_full", "precision_full"])


In [ ]:
threshold = 0.24
y_pred = (p_uncal >= threshold).astype(int)

cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix (rows=true, cols=pred):")
print(cm)

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

In [ ]:
def confusion_at_threshold(
        y_true,
        p_pred,
        threshold: float
):
    """
    Compute confusion matrix + key rates at a chosen probability threshold.
    """

    y_hat = (p_pred >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_hat).ravel()

    denominator = np.sqrt(
        (tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))

    phi = (tp * tn - fp * fn) / denominator if denominator > 0 else np.nan

    metrics = {
        "threshold": threshold,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "recall_full": tp / (tp + fn) if (tp + fn) > 0 else np.nan,
        "precision_full": tp / (tp + fp) if (tp + fp) > 0 else np.nan,
        "false_positive_rate": fp / (fp + tn) if (fp + tn) > 0 else np.nan,
        "false_negative_rate": fn / (fn + tp) if (fn + tp) > 0 else np.nan,
        "TNR": tn / (fp + tn) if (fp + tn) > 0 else np.nan,
        "NPV": tn / (tn + fn) if (tn + fn) > 0 else np.nan,
        "FOR": fn / (tn + fn) if (tn + fn) > 0 else np.nan,
        "FDR": fp / (tp + fp) if (tp + fp) > 0 else np.nan,
        "phi": phi

    }

    return metrics, y_hat

In [ ]:
range(.05, 1, .1)

In [ ]:
threshold = range(5, 100)

df = pd.DataFrame()
for t in threshold:
    t = t / 100
    metrics, y_pred = confusion_at_threshold(
        y_true=y_test,
        p_pred=p_uncal,
        threshold=t
    )

    df = pd.concat([df, pd.DataFrame([metrics])], ignore_index=True)


In [ ]:
df.to_csv("1.csv")

In [ ]:
print("Confusion Matrix (rows=true, cols=pred):")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, digits=3))
